# 04 — Segmentação de Produtos (KMeans)

Este notebook executa segmentação via KMeans usando um pipeline (Imputer → StandardScaler → KMeans), calcula Elbow + Silhouette e exporta o modelo e a base com clusters.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
MODELS_DIR = REPORTS_DIR / 'models'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from src.clustering import fit_kmeans_pipeline, name_clusters, save_pipeline, score_k_range


C:\Users\flavi\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Carregar base com PSI

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'base_produtos_psi.csv')
df.shape


(1351, 18)

## Elbow + Silhouette (k=2..10)

In [3]:
features = ['discounted_price_clean','discount_pct_clean','rating_clean','rating_count_clean','PSI']
metrics = score_k_range(df, features=features, k_values=range(2, 11))
metrics


,k,inertia,silhouette
0,2,5227.615335,0.248314
1,3,4343.505343,0.264361
2,4,3391.267647,0.301007
3,5,2645.692081,0.310216
4,6,2401.261380,0.309004
5,7,2141.523809,0.254813
6,8,1948.960951,0.259385
7,9,1738.912645,0.267061
8,10,1631.758305,0.251377


In [4]:
plt.figure(figsize=(10,4))
plt.plot(metrics['k'], metrics['inertia'], marker='o')
plt.title('Elbow Curve (Inertia)')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kmeans_elbow.png', dpi=160)
plt.close()

plt.figure(figsize=(10,4))
plt.plot(metrics['k'], metrics['silhouette'], marker='o')
plt.title('Silhouette Score by k')
plt.xlabel('k')
plt.ylabel('Silhouette')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kmeans_silhouette.png', dpi=160)
plt.close()


## Treinar modelo final e exportar artefatos

In [5]:
best_k = int(metrics.sort_values('silhouette', ascending=False).iloc[0]['k'])
best_k


5

In [6]:
artifacts = fit_kmeans_pipeline(df, features=features, n_clusters=best_k)
df_out = df.copy()
df_out['cluster'] = artifacts.labels.values
df_out = pd.concat([df_out, artifacts.pca_2d], axis=1)
cluster_names = name_clusters(df_out, cluster_col='cluster')
df_out['cluster_name'] = df_out['cluster'].map(cluster_names)
df_out[['cluster','cluster_name']].value_counts().reset_index(name='n').head(10)


,cluster,cluster_name,n
0,4,barato com alto desconto,576
1,0,segmento misto,416
2,1,segmento misto,248
3,3,premium bem avaliado,79
4,2,segmento misto,32


In [7]:
save_pipeline(artifacts.pipeline, str(MODELS_DIR / 'kmeans_pipeline.joblib'))
df_out.to_csv(PROCESSED_DIR / 'base_produtos_com_clusters.csv', index=False)
df_out.to_csv(REPORTS_DIR / 'base_produtos_com_clusters.csv', index=False)
(MODELS_DIR / 'kmeans_pipeline.joblib', PROCESSED_DIR / 'base_produtos_com_clusters.csv')


(WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/reports/models/kmeans_pipeline.joblib'),
 WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/data/processed/base_produtos_com_clusters.csv'))